DATA LOADING & PREPARATION

In [3]:
#IMPORTING LIBRARIES
import pandas as pd
import numpy as np

def clean_and_merge_chicago_data(crashes_path, vehicles_path, people_path):
    print("Loading datasets...")
    crashes = pd.read_csv(crashes_path)
    vehicles = pd.read_csv(vehicles_path)
    people = pd.read_csv(people_path)
    
    
    return crashes, vehicles, people

ModuleNotFoundError: No module named 'pandas'

In [ ]:
import numpy as np
# The "Emergency Hack" to fix the version conflict
np.int = int
np.float = float
np.bool = bool

In [ ]:
# Defining my paths (update these to my actual file locations)
c_path = "Crashes.csv"
v_path = "Vehicles.csv"
p_path = "People.csv"

#calling the function and seeing the variables it returns
# Making sure the function has 'return crashes, vehicles, people' at the end!
crashes, vehicles, people = clean_and_merge_chicago_data(c_path, v_path, p_path)

Loading datasets...


c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\IPython\core\interactiveshell.py:3337: DtypeWarning: Columns (10) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):
c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\IPython\core\interactiveshell.py:3337: DtypeWarning: Columns (3,18,20,38,39,40,41,43,47,48,49,52,54,57,58,60,64,70) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):
c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\IPython\core\interactiveshell.py:3337: DtypeWarning: Columns (28) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):


In [ ]:
#checking if the data is loaded as it was quite heavy
people.head()

,PERSON_ID,PERSON_TYPE,CRASH_RECORD_ID,VEHICLE_ID,CRASH_DATE,SEAT_NO,CITY,STATE,ZIPCODE,SEX,...,EMS_RUN_NO,DRIVER_ACTION,DRIVER_VISION,PHYSICAL_CONDITION,PEDPEDAL_ACTION,PEDPEDAL_VISIBILITY,PEDPEDAL_LOCATION,BAC_RESULT,BAC_RESULT VALUE,CELL_PHONE_USE
0,O2270763,DRIVER,5062bad4c3bf18e33e21532338a49fa0226fd13d1a871a...,2165499.0,03/24/2026 01:30:00 AM,NaN,KALAMAZOO,MI,49009,M,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
1,O2270759,DRIVER,8c793b331e24b63168d942f7c0d4026c260659ea7146ec...,2165492.0,03/24/2026 01:17:00 AM,NaN,NaN,NaN,NaN,X,...,NaN,UNKNOWN,UNKNOWN,UNKNOWN,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
2,O2270760,DRIVER,8c793b331e24b63168d942f7c0d4026c260659ea7146ec...,2165494.0,03/24/2026 01:17:00 AM,NaN,BRIDGEVIEW,IL,60455,M,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
3,O2270742,DRIVER,442e755490927c117df2c667e7955aed33703fe1882ba6...,2165479.0,03/23/2026 11:45:00 PM,NaN,CHICAGO,IL,60609,F,...,NaN,NONE,NOT OBSCURED,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
4,P499444,PASSENGER,442e755490927c117df2c667e7955aed33703fe1882ba6...,2165479.0,03/23/2026 11:45:00 PM,3.0,CHICAGO,IL,60625,F,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# --- 1. PREPARE PEOPLE (Focus on Drivers) ---
print("Processing People data...")
# Filter for Drivers as they are most relevant to contributory causes
drivers = people[people['PERSON_TYPE'] == 'DRIVER'].copy()

# Aggregate driver attributes per crash record
drivers_agg = drivers.groupby('CRASH_RECORD_ID').agg({
    'AGE': 'mean',
    'SEX': lambda x: x.mode()[0] if not x.mode().empty else 'U',
    'PHYSICAL_CONDITION': lambda x: x.mode()[0] if not x.mode().empty else 'UNKNOWN'
}).reset_index()

drivers_agg.columns = ['CRASH_RECORD_ID', 'DRIVER_AGE_MEAN', 'DRIVER_SEX', 'DRIVER_PHYS_COND']

Processing People data...


In [ ]:
#checking if all is ok on the above
print(drivers_agg.shape)
drivers_agg.head()

(1035548, 4)


,CRASH_RECORD_ID,DRIVER_AGE_MEAN,DRIVER_SEX,DRIVER_PHYS_COND
0,000013b0123279411e0ec856dae95ab9f0851764350b7f...,51.0,F,NORMAL
1,00002c0771fb6f2c70ba775b7f6b501608cadea85c1dd1...,35.0,F,NORMAL
2,000043c6564ec4d54bc4efd957d97ca97f38a965dd64b4...,NaN,X,UNKNOWN
3,00005696946846c8b8a1d378dba4e2a5ed84a9b2876fe0...,54.0,M,NORMAL
4,000070ed7a6357c3298f5edc6fb7d5ce925a10f46660f3...,47.5,M,NORMAL


In [ ]:
# --- 2. PREPARE VEHICLES ---
print("Processing Vehicles data...")
# Aggregate vehicle attributes to ensure one row per crash record
vehicles_agg = vehicles.groupby('CRASH_RECORD_ID').agg({
    'VEHICLE_DEFECT': lambda x: 'DEFECT' if any(val != 'NONE' and val != 'UNKNOWN' for val in x) else 'NONE',
    'VEHICLE_TYPE': lambda x: x.mode()[0] if not x.mode().empty else 'UNKNOWN',
    'NUM_PASSENGERS': 'sum'
}).reset_index()

Processing Vehicles data...


In [ ]:
#3.MERGING DATASETS 
print("Merging datasets...")
df = crashes.merge(vehicles_agg, on='CRASH_RECORD_ID', how='left')
df = df.merge(drivers_agg, on='CRASH_RECORD_ID', how='left')

Merging datasets...


In [ ]:
# Checking the results to confirm if merging was successful
print(f"Columns in merged dataset: {df.columns.tolist()}")
print(f"Total rows: {len(df)}")
df.head()

Columns in merged dataset: ['CRASH_RECORD_ID', 'CRASH_DATE_EST_I', 'CRASH_DATE', 'POSTED_SPEED_LIMIT', 'TRAFFIC_CONTROL_DEVICE', 'DEVICE_CONDITION', 'WEATHER_CONDITION', 'LIGHTING_CONDITION', 'FIRST_CRASH_TYPE', 'TRAFFICWAY_TYPE', 'LANE_CNT', 'ALIGNMENT', 'ROADWAY_SURFACE_COND', 'ROAD_DEFECT', 'REPORT_TYPE', 'CRASH_TYPE', 'INTERSECTION_RELATED_I', 'NOT_RIGHT_OF_WAY_I', 'HIT_AND_RUN_I', 'DAMAGE', 'DATE_POLICE_NOTIFIED', 'PRIM_CONTRIBUTORY_CAUSE', 'SEC_CONTRIBUTORY_CAUSE', 'STREET_NO', 'STREET_DIRECTION', 'STREET_NAME', 'BEAT_OF_OCCURRENCE', 'PHOTOS_TAKEN_I', 'STATEMENTS_TAKEN_I', 'DOORING_I', 'WORK_ZONE_I', 'WORK_ZONE_TYPE', 'WORKERS_PRESENT_I', 'NUM_UNITS', 'MOST_SEVERE_INJURY', 'INJURIES_TOTAL', 'INJURIES_FATAL', 'INJURIES_INCAPACITATING', 'INJURIES_NON_INCAPACITATING', 'INJURIES_REPORTED_NOT_EVIDENT', 'INJURIES_NO_INDICATION', 'INJURIES_UNKNOWN', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH', 'LATITUDE', 'LONGITUDE', 'LOCATION', 'VEHICLE_DEFECT', 'VEHICLE_TYPE', 'NUM_PASSENGERS', 

,CRASH_RECORD_ID,CRASH_DATE_EST_I,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,...,CRASH_MONTH,LATITUDE,LONGITUDE,LOCATION,VEHICLE_DEFECT,VEHICLE_TYPE,NUM_PASSENGERS,DRIVER_AGE_MEAN,DRIVER_SEX,DRIVER_PHYS_COND
0,5062bad4c3bf18e33e21532338a49fa0226fd13d1a871a...,NaN,03/24/2026 01:30:00 AM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,"DARKNESS, LIGHTED ROAD",OTHER OBJECT,DIVIDED - W/MEDIAN (NOT RAISED),...,3,41.874310,-87.666641,POINT (-87.666640882211 41.874309624504),NONE,SPORT UTILITY VEHICLE (SUV),0.0,30.0,M,NORMAL
1,8c793b331e24b63168d942f7c0d4026c260659ea7146ec...,NaN,03/24/2026 01:17:00 AM,30,TRAFFIC SIGNAL,UNKNOWN,CLEAR,"DARKNESS, LIGHTED ROAD",REAR END,FOUR WAY,...,3,41.786437,-87.683975,POINT (-87.683975246481 41.786437174774),NONE,PASSENGER,0.0,19.0,M,NORMAL
2,442e755490927c117df2c667e7955aed33703fe1882ba6...,NaN,03/23/2026 11:45:00 PM,10,NO CONTROLS,NO CONTROLS,CLEAR,DARKNESS,FIXED OBJECT,ALLEY,...,3,41.828339,-87.623042,POINT (-87.623042178584 41.82833907934),NONE,PASSENGER,1.0,31.0,F,NORMAL
3,79954fa483cb69371e38b0fd3222ebfdb7b3a904851deb...,NaN,03/23/2026 11:30:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,"DARKNESS, LIGHTED ROAD",TURNING,T-INTERSECTION,...,3,41.751020,-87.615102,POINT (-87.615102245534 41.751020470682),NONE,BUS OVER 15 PASS.,0.0,52.0,M,NORMAL
4,0a7917f8e7b045973ec2830ed7c3e8484b4725e6f88309...,NaN,03/23/2026 11:15:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,"DARKNESS, LIGHTED ROAD",TURNING,CENTER TURN LANE,...,3,41.886639,-87.644126,POINT (-87.644126426269 41.886638946448),NONE,PASSENGER,0.0,31.0,F,NORMAL


TARGET GROUPING

In [ ]:
#creating a column for the primary contributory cause
print("Grouping target variable...")
cause_map = {
        'FAILING TO YIELD RIGHT-OF-WAY': 'DRIVER_ERROR',
        'FOLLOWING TOO CLOSELY': 'DRIVER_ERROR',
        'IMPROPER OVERTAKING/PASSING': 'DRIVER_ERROR',
        'IMPROPER TURNING/NO SIGNAL': 'DRIVER_ERROR',
        'IMPROPER LANE USAGE': 'DRIVER_ERROR',
        'IMPROPER BACKING': 'DRIVER_ERROR',
        'DRIVING SKILLS/KNOWLEDGE/EXPERIENCE': 'DRIVER_ERROR',
        'OPERATING VEHICLE IN ERRATIC, RECKLESS, CARELESS, NEGLIGENT OR AGGRESSIVE MANNER': 'DRIVER_ERROR',
        'FAILING TO REDUCE SPEED TO AVOID CRASH': 'DRIVER_ERROR',
        'DISREGARDING TRAFFIC SIGNALS': 'VIOLATION',
        'DISREGARDING STOP SIGN': 'VIOLATION',
        'DISREGARDING OTHER TRAFFIC SIGNS': 'VIOLATION',
        'DISREGARDING ROAD MARKINGS': 'VIOLATION',
        'EXCEEDING SAFE SPEED LIMIT': 'VIOLATION',
        'EXCEEDING POSTED SPEED LIMIT': 'VIOLATION',
        'UNDER THE INFLUENCE OF ALCOHOL/DRUGS (USE WHEN ARREST IS EFFECTED)': 'INTOXICATION',
        'HAD BEEN DRINKING (USE WHEN ARREST IS NOT MADE)': 'INTOXICATION',
        'DISTRACTION - FROM INSIDE VEHICLE': 'DISTRACTION',
        'DISTRACTION - FROM OUTSIDE VEHICLE': 'DISTRACTION',
        'DISTRACTION - OTHER ELECTRONIC DEVICE (NAVIGATION DEVICE, DVD PLAYER, ETC.)': 'DISTRACTION',
        'CELL PHONE USE OTHER THAN TEXTING': 'DISTRACTION',
        'TEXTING': 'DISTRACTION',
        'WEATHER': 'ENVIRONMENT',
        'ROAD ENGINEERING/SURFACE/MARKING DEFECTS': 'ENVIRONMENT',
        'ROAD CONSTRUCTION/MAINTENANCE': 'ENVIRONMENT',
        'VISION OBSCURED (SIGNS, TREE LIMBS, BUILDINGS, ETC.)': 'ENVIRONMENT',
        'ANIMAL': 'ENVIRONMENT',
        'EQUIPMENT - VEHICLE CONDITION': 'VEHICLE_DEFECT'
    }
    
df['TARGET_CAUSE'] = df['PRIM_CONTRIBUTORY_CAUSE'].map(cause_map).fillna('OTHER_UNKNOWN')
    
    # Focus the model by removing entries where the cause is unknown or not applicable
df = df[df['TARGET_CAUSE'] != 'OTHER_UNKNOWN']

Grouping target variable...


In [ ]:
# Checking how many of each category we have now
print(df['TARGET_CAUSE'].value_counts())

DRIVER_ERROR      470719
VIOLATION          34368
ENVIRONMENT        26801
DISTRACTION        13082
VEHICLE_DEFECT      6319
INTOXICATION        5798
Name: TARGET_CAUSE, dtype: int64


In [ ]:
#FINAL CLEANING(Dimensionality Reduction)
# Drop columns that are more than 50% empty
limit = len(df) * 0.5
df = df.dropna(thresh=limit, axis=1)
    
print(f"Final dataset shape: {df.shape}")

Final dataset shape: (557087, 44)


In [ ]:
# --- CONSOLIDATED MERGE & CLEAN 

# 1. Re-aggregate Drivers
drivers = people[people['PERSON_TYPE'] == 'DRIVER'].copy()
drivers_agg = drivers.groupby('CRASH_RECORD_ID').agg({
    'AGE': 'mean',
    'SEX': lambda x: x.mode()[0] if not x.mode().empty else 'U',
    'PHYSICAL_CONDITION': lambda x: x.mode()[0] if not x.mode().empty else 'UNKNOWN'
}).reset_index()
drivers_agg.columns = ['CRASH_RECORD_ID', 'DRIVER_AGE_MEAN', 'DRIVER_SEX', 'DRIVER_PHYS_COND']

# 2. Re-aggregate Vehicles
vehicles_agg = vehicles.groupby('CRASH_RECORD_ID').agg({
    'VEHICLE_DEFECT': lambda x: 'DEFECT' if any(val != 'NONE' and val != 'UNKNOWN' for val in x) else 'NONE',
    'VEHICLE_TYPE': lambda x: x.mode()[0] if not x.mode().empty else 'UNKNOWN',
    'NUM_PASSENGERS': 'sum'
}).reset_index()

# 3. Perform the Merge on the original crashes dataframe
df = crashes.merge(vehicles_agg, on='CRASH_RECORD_ID', how='left')
df = df.merge(drivers_agg, on='CRASH_RECORD_ID', how='left')

# 4. Re-apply the Target Grouping 
df['TARGET_CAUSE'] = df['PRIM_CONTRIBUTORY_CAUSE'].map(cause_map).fillna('OTHER_UNKNOWN')
df_cleaned = df[df['TARGET_CAUSE'] != 'OTHER_UNKNOWN'].copy()

# 5. VERIFICATION CHECK (This should print True for all)
print(f"Has DRIVER_SEX: {'DRIVER_SEX' in df_cleaned.columns}")
print(f"Has VEHICLE_TYPE: {'VEHICLE_TYPE' in df_cleaned.columns}")
print(f"Final Row Count: {len(df_cleaned)}")

Has DRIVER_SEX: True
Has VEHICLE_TYPE: True
Final Row Count: 557087


FEATURE SEPARATION

In [ ]:
#separating features into categorical and numerical features
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
# We select features that are available at the time of the crash
categorical_features = [
    'WEATHER_CONDITION', 'LIGHTING_CONDITION', 'FIRST_CRASH_TYPE', 
    'TRAFFICWAY_TYPE', 'ALIGNMENT', 'ROADWAY_SURFACE_COND', 
    'DRIVER_SEX', 'VEHICLE_TYPE', 'DRIVER_PHYS_COND'
]

numeric_features = [
    'POSTED_SPEED_LIMIT', 'NUM_UNITS', 'CRASH_HOUR', 
    'CRASH_DAY_OF_WEEK', 'CRASH_MONTH', 'DRIVER_AGE_MEAN', 'NUM_PASSENGERS'
]

X = df_cleaned[categorical_features + numeric_features]
y = df_cleaned['TARGET_CAUSE']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

Training set size: (445669, 16)
Testing set size: (111418, 16)


In [ ]:
#Build the Preprocessing Pipeline
#Data Preparation, we use a ColumnTransformer. This automates the scaling of numbers and the encoding of categories.

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Pipeline for numeric data: Impute missing values then scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Pipeline for categorical data: Impute with 'missing' then One-Hot Encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='UNKNOWN')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# 3. Combine into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("Preprocessing Pipeline ready.")

Preprocessing Pipeline ready.


    BASELINE DECISION TREE

In [ ]:
#The Baseline Decision Tree (White-Box)
#We now bundle the preprocessor with a DecisionTreeClassifier. This is our White-Box model, which is intrinsically interpretable.


from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Create the full pipeline
baseline_dt = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced'))
])

# Fit the model
baseline_dt.fit(X_train, y_train)

# Predict and Evaluate
y_pred = baseline_dt.predict(X_test)

print("--- Baseline Decision Tree Performance ---")
print(classification_report(y_test, y_pred))

--- Baseline Decision Tree Performance ---
                precision    recall  f1-score   support

   DISTRACTION       0.04      0.64      0.07      2616
  DRIVER_ERROR       0.93      0.14      0.24     94144
   ENVIRONMENT       0.14      0.61      0.23      5360
  INTOXICATION       0.08      0.83      0.15      1160
VEHICLE_DEFECT       0.00      0.00      0.00      1264
     VIOLATION       0.29      0.68      0.41      6874

      accuracy                           0.21    111418
     macro avg       0.25      0.49      0.18    111418
  weighted avg       0.81      0.21      0.24    111418



MEANING OF RESULTS

Our baseline model is weak since we have class imbalance problem.
The overall accuracy is 21%.
The weighted average is 0.81 for the precision meaning it is only predicting the dominant class, hence the weakness of the baseline model

RANDOM FOREST-ITERATION 1

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Define the Random Forest Pipeline
# We use 'balanced_subsample' which is great for imbalanced data like ours
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100, 
        random_state=42, 
        class_weight='balanced_subsample',
        n_jobs=-1
    ))
])

# Fit and Predict
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

print("--- Iteration 1: Random Forest Performance ---")
print(classification_report(y_test, y_pred_rf))

--- Iteration 1: Random Forest Performance ---
                precision    recall  f1-score   support

   DISTRACTION       0.19      0.01      0.01      2616
  DRIVER_ERROR       0.87      0.98      0.92     94144
   ENVIRONMENT       0.63      0.28      0.38      5360
  INTOXICATION       0.75      0.65      0.69      1160
VEHICLE_DEFECT       0.22      0.01      0.01      1264
     VIOLATION       0.50      0.18      0.27      6874

      accuracy                           0.86    111418
     macro avg       0.53      0.35      0.38    111418
  weighted avg       0.81      0.86      0.82    111418



MEANING OF RESULTS

The Random Forest model significantly outperforms the baseline Decision Tree. 

While the Decision Tree achieved only 21% accuracy and showed unstable predictions with poor recall for the majority class, the Random Forest improved accuracy to 86% and demonstrated strong performance, particularly for the DRIVER_ERROR class with a recall of 0.98. 

Additionally, the Random Forest produced more balanced and reliable predictions, as reflected in the higher F1-scores. This improvement is due to the ensemble nature of Random Forest, which reduces overfitting and increases generalization

TUNED RANDOM FOREST-ITERATION 2

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV

# 1. The Patch for your environment
np.int = int
np.float = float
np.bool = bool

# 2. Re-define the grid to avoid NameError
param_grid_rf = {
    'classifier__max_depth': [5, 10, None],
    'classifier__n_estimators': [100, 200],
    'classifier__min_samples_leaf': [1, 5]
}

# 3. Initialize and Run (n_jobs=1 for stability)
# Using cv=3 to make it run a bit faster since we're on one core
rf_grid_search = GridSearchCV(rf_pipeline, param_grid_rf, cv=3, scoring='f1_macro', n_jobs=1)

print("Starting training... this might take a minute...")
rf_grid_search.fit(X_train, y_train)
print("Training complete!")

# 4. Feature Importance with Version-Safe Extraction
best_model = rf_grid_search.best_estimator_
preprocessor = best_model.named_steps['preprocessor']

# Get names from the categorical part
cat_transformer = preprocessor.named_steps['cat'] # Adjust 'cat' to your pipeline's name
if hasattr(cat_transformer, 'get_feature_names_out'):
    cat_features = cat_transformer.get_feature_names_out()
else:
    # Older versions use get_feature_names()
    cat_features = cat_transformer.get_feature_names()

# Combine with numerical names (assuming they didn't change)
num_features = ["age", "posted_speed_limit", "weather_condition"] # Put your actual num columns here
all_feature_names = list(num_features) + list(cat_features)

# Create the importance DF
importances = best_model.named_steps['classifier'].feature_importances_
feat_df = pd.DataFrame({'Feature': all_feature_names, 'Importance': importances})
print(feat_df.sort_values(by='Importance', ascending=False).head(10))

Starting training... this might take a minute...


exception calling callback for <Future at 0x1f3c6a3a190 state=finished raised BrokenProcessPool>
joblib.externals.loky.process_executor._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\joblib\externals\loky\process_executor.py", line 404, in _process_worker
    call_item = call_queue.get(block=True, timeout=timeout)
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\multiprocessing\queues.py", line 116, in get
    return _ForkingPickler.loads(res)
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\sklearn\__init__.py", line 80, in <module>
    from .base import clone
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\sklearn\base.py", line 21, in <module>
    from .utils import _IS_32BIT
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\sklearn\utils\__init__.py", line 23, in <module>
    from .class_weight import compute_class_weight, compute_sample_weight
  File "

Training complete!


AttributeError: 'ColumnTransformer' object has no attribute 'named_steps'

In [ ]:
"""
Updating GridSearchCV code. This enables the code to run without errors.

"""
# Change n_jobs=-1 to n_jobs=None (or 1)
rf_grid_search = GridSearchCV(
    rf_pipeline, 
    param_grid_rf, 
    cv=3, 
    scoring='f1_macro', 
    n_jobs=None  # This avoids the "BrokenProcessPool" error
)

rf_grid_search.fit(X_train, y_train)

NameError: name 'param_grid_rf' is not defined

In [ ]:
"""1. The "Safety First" Fix (Highly Recommended)
Change n_jobs=-1 to n_jobs=1.

Parallel processing is great, but it hides the real errors behind that messy BrokenProcessPool message. By running it on a single core, you bypass the serialization (pickling) issue entirely.

Python"""
# Change n_jobs from -1 to 1
rf_grid_search = GridSearchCV(rf_pipeline, param_grid_rf, cv=3, scoring='f1_macro', n_jobs=1)
rf_grid_search.fit(X_train, y_train)

exception calling callback for <Future at 0x2ddc63829d0 state=finished raised BrokenProcessPool>
joblib.externals.loky.process_executor._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\joblib\externals\loky\process_executor.py", line 404, in _process_worker
    call_item = call_queue.get(block=True, timeout=timeout)
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\multiprocessing\queues.py", line 116, in get
    return _ForkingPickler.loads(res)
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\sklearn\__init__.py", line 80, in <module>
    from .base import clone
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\sklearn\base.py", line 21, in <module>
    from .utils import _IS_32BIT
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\sklearn\utils\__init__.py", line 23, in <module>
    from .class_weight import compute_class_weight, compute_sample_weight
  File "

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['POSTED_SPEED_LIMIT',
                                                                          'NUM_UNITS',
                                                                          'CRASH_HOUR',
                                                                          'CRASH_DAY_OF_WEEK',
                                          

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the search grid
# We tune 'max_depth' to control complexity and 'n_estimators' for the number of trees
param_grid_rf = {
    'classifier__max_depth': [5, 10, None],
    'classifier__n_estimators': [100, 200],
    'classifier__min_samples_leaf': [1, 5]
}

# Initialize and run Grid Search
# We use 'f1_macro' to ensure we care about the rare classes (Intoxication, Defects)
rf_grid_search = GridSearchCV(rf_pipeline, param_grid_rf, cv=3, scoring='f1_macro', n_jobs=1)
rf_grid_search.fit(X_train, y_train)

# Final Results
print(f"Best RF Parameters: {rf_grid_search.best_params_}")
y_pred_tuned_rf = rf_grid_search.predict(X_test)

print("\n--- Iteration 2: Tuned Random Forest Performance ---")
print(classification_report(y_test, y_pred_tuned_rf))

exception calling callback for <Future at 0x2ddc609afd0 state=finished raised BrokenProcessPool>
joblib.externals.loky.process_executor._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\joblib\externals\loky\process_executor.py", line 404, in _process_worker
    call_item = call_queue.get(block=True, timeout=timeout)
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\multiprocessing\queues.py", line 116, in get
    return _ForkingPickler.loads(res)
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\sklearn\__init__.py", line 80, in <module>
    from .base import clone
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\sklearn\base.py", line 21, in <module>
    from .utils import _IS_32BIT
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\sklearn\utils\__init__.py", line 23, in <module>
    from .class_weight import compute_class_weight, compute_sample_weight
  File "

Best RF Parameters: {'classifier__max_depth': 5, 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 100}

--- Iteration 2: Tuned Random Forest Performance ---
                precision    recall  f1-score   support

   DISTRACTION       0.04      0.36      0.07      2616
  DRIVER_ERROR       0.94      0.37      0.53     94144
   ENVIRONMENT       0.18      0.62      0.28      5360
  INTOXICATION       0.15      0.86      0.25      1160
VEHICLE_DEFECT       0.06      0.40      0.11      1264
     VIOLATION       0.26      0.70      0.38      6874

      accuracy                           0.40    111418
     macro avg       0.27      0.55      0.27    111418
  weighted avg       0.82      0.40      0.49    111418



MEANING OF RESULTS

The tuned Random Forest model reduces bias toward the majority class and improves detection of minority classes such as DISTRACTION and VEHICLE_DEFECT, as shown by higher recall values. 

However, this comes at the cost of lower overall accuracy (40% compared to 86%) and reduced performance on the dominant DRIVER_ERROR class. Tuning was necessary to create a more balanced model that does not ignore rare but important classes, even though it sacrifices some accuracy.

MODEL INTERPRETABILITY

We have chosen the tuned Random Forest as it has proven to be the most reliable model.

Although the original Random Forest achieved higher accuracy, it failed to capture minority classes effectively. Therefore, the tuned model was preferred for interpretability.

In [ ]:
pipeline = rf_grid_search.best_estimator_

In [ ]:
#choosing the model to interpreted
model = pipeline.named_steps['classifier']

In [ ]:
# 1. Run the search on a single core for stability
rf_grid_search = GridSearchCV(rf_pipeline, param_grid_rf, cv=3, scoring='f1_macro', n_jobs=1)
rf_grid_search.fit(X_train, y_train)

# 2. Extract feature names using the older method name
# We use a try/except block to handle the version difference automatically
preprocessor = rf_grid_search.best_estimator_.named_steps['preprocessor']

try:
    feature_names = preprocessor.get_feature_names_out()
except AttributeError:
    # This is for your current version of scikit-learn
    feature_names = preprocessor.get_feature_names()

# 3. View the results
importances = rf_grid_search.best_estimator_.named_steps['classifier'].feature_importances_
feat_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
print(feat_df.sort_values(by='Importance', ascending=False).head(10))

exception calling callback for <Future at 0x1f3c6a2aca0 state=finished raised BrokenProcessPool>
joblib.externals.loky.process_executor._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\joblib\externals\loky\process_executor.py", line 404, in _process_worker
    call_item = call_queue.get(block=True, timeout=timeout)
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\multiprocessing\queues.py", line 116, in get
    return _ForkingPickler.loads(res)
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\sklearn\__init__.py", line 80, in <module>
    from .base import clone
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\sklearn\base.py", line 21, in <module>
    from .utils import _IS_32BIT
  File "c:\Users\morin\anaconda3\envs\learn-env\lib\site-packages\sklearn\utils\__init__.py", line 23, in <module>
    from .class_weight import compute_class_weight, compute_sample_weight
  File "

AttributeError: Transformer num (type Pipeline) does not provide get_feature_names.

In [ ]:
for name, transformer, cols in preprocessor.transformers_:
    
    if transformer == 'drop' or transformer is None:
        continue
    
    # If transformer is OneHotEncoder or similar
    if hasattr(transformer, 'get_feature_names'):
        new_features = transformer.get_feature_names(cols)
        feature_names.extend(new_features)
    else:
        # numeric columns (unchanged)
        feature_names.extend(cols)

NameError: name 'preprocessor' is not defined

In [ ]:
# Extract the trained Random Forest from pipeline
model = rf_grid_search.best_estimator_.named_steps['classifier']

# Feature importance
importances = model.feature_importances_

# Dataframe
feat_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feat_df.head(10))

# Plot
plt.figure()
plt.barh(feat_df['Feature'][:10], feat_df['Importance'][:10])
plt.gca().invert_yaxis()
plt.title("Top 10 Feature Importances (Tuned RF)")
plt.show()

ValueError: arrays must all be same length